In [1]:
import torch
import torch.nn as nn
import numpy as np

from pathlib import Path
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix

In [2]:
class DigitCNN(nn.Module):

    def __init__(self):
        super(DigitCNN, self).__init__()

        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)

        self.fc1 = nn.Linear(16 * 4 * 4, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):

        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)

        x = x.view(x.size(0), -1)

        x = self.fc1(x)
        x = self.relu(x)

        x = self.fc2(x)

        return x

In [3]:
# Load test data
data = np.load("data/digit_splits.npz")

X_test = data["X_test"]
y_test = data["y_test"]

# Convert test data into PyTorch tensors
X_test = torch.tensor(
    X_test,
    dtype=torch.float32
).unsqueeze(1)

y_test = torch.tensor(
    y_test,
    dtype=torch.long
)

# Create CNN model
cnn_model = DigitCNN()

# Load saved CNN weights
cnn_model.load_state_dict(
    torch.load("models/cnn_model.pth", map_location="cpu")
)

# Set evaluation mode
cnn_model.eval()

print("CNN model loaded successfully!")
print("Test data shape:", X_test.shape)

CNN model loaded successfully!
Test data shape: torch.Size([270, 1, 8, 8])


In [4]:
# Make CNN predictions
with torch.no_grad():

    outputs = cnn_model(X_test)

    predictions = torch.argmax(outputs, dim=1)

# Convert predictions to NumPy
y_true = y_test.numpy()
cnn_predictions = predictions.numpy()

# Calculate accuracy
cnn_accuracy = accuracy_score(
    y_true,
    cnn_predictions
)

# Calculate Macro F1-score
cnn_f1 = f1_score(
    y_true,
    cnn_predictions,
    average="macro"
)

print("CNN Test Accuracy:", cnn_accuracy * 100)
print("CNN Macro F1-score:", cnn_f1)

CNN Test Accuracy: 98.14814814814815
CNN Macro F1-score: 0.9811430461632155


In [5]:
# Calculate CNN confusion matrix
cnn_cm = confusion_matrix(
    y_true,
    cnn_predictions
)

print("CNN Confusion Matrix:")
print(cnn_cm)

CNN Confusion Matrix:
[[27  0  0  0  0  0  0  0  0  0]
 [ 0 28  0  0  0  0  0  0  0  0]
 [ 0  0 26  0  0  0  0  0  0  0]
 [ 0  0  0 28  0  0  0  0  0  0]
 [ 0  0  0  0 26  0  1  0  0  0]
 [ 0  0  0  0  0 27  0  0  0  0]
 [ 0  0  0  0  0  0 26  0  1  0]
 [ 0  0  0  0  0  0  0 27  0  0]
 [ 0  2  0  0  0  0  0  1 23  0]
 [ 0  0  0  0  0  0  0  0  0 27]]


In [13]:
import json

with open("models/vit_metrics.json", "r") as f:
    vit_metrics = json.load(f)

vit_accuracy = vit_metrics["vit_accuracy"]
vit_f1 = vit_metrics["vit_f1"]

In [14]:

print("\n===== MODEL COMPARISON =====")

print("Model       Accuracy       Macro F1")

print(
    f"CNN         {cnn_accuracy * 100:.2f}%"
    f"          {cnn_f1:.4f}"
)

print(
    f"ViT         {vit_accuracy * 100:.2f}%"
    f"          {vit_f1:.4f}"
)


===== MODEL COMPARISON =====
Model       Accuracy       Macro F1
CNN         98.15%          0.9811
ViT         94.07%          0.9408


In [15]:
from collections import Counter

# Find most common digit in training labels
counts = Counter(data["y_train"])

majority_digit, majority_count = counts.most_common(1)[0]

# Predict the same digit for every test image
baseline_predictions = np.full(
    len(y_true),
    majority_digit
)

# Calculate baseline accuracy
baseline_accuracy = accuracy_score(
    y_true,
    baseline_predictions
)

print("Most common digit:", majority_digit)
print("Baseline Accuracy:", baseline_accuracy * 100)

Most common digit: 3
Baseline Accuracy: 10.37037037037037
